# TinyLLM Phase 2 on Vertex AI Workbench
Run **GPTQ vs AWQ** on **Phi-3-mini**, export **GGUF**, and benchmark CPU with **llama.cpp**.

_Last generated: 2025-10-23 02:12:07_

## 0) Environment (Python & system info)

In [ ]:

!python -V
import psutil, platform, torch, sys
print("RAM (GB):", round(psutil.virtual_memory().total/1e9,2))
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())


## 1) Configure GCP (Optional but recommended)
- Set your project, region, and a GCS bucket for artifacts.
- If you're already authenticated in a Vertex AI Workbench notebook, you can skip explicit auth.

In [ ]:

PROJECT_ID = "YOUR_PROJECT_ID"
REGION = "us-central1"   # or your preferred region
BUCKET = "gs://YOUR_BUCKET"  # must exist
EXPERIMENT = "tinyllm-phase2"

import os, time, pathlib
ts = time.strftime("%Y%m%d-%H%M%S")
ARTIFACTS = f"{BUCKET}/{EXPERIMENT}/{ts}"

print("PROJECT_ID:", PROJECT_ID)
print("REGION:", REGION)
print("ARTIFACTS:", ARTIFACTS)


In [ ]:

# Uncomment if needed in Workbench (usually already authed)
# from google.colab import auth
# auth.authenticate_user()

# !gcloud config set project $PROJECT_ID


## 2) Install dependencies (pinned)

In [ ]:

%%bash
pip -q install --upgrade pip
pip -q install transformers==4.43.4 accelerate==0.33.0 datasets==2.21.0 evaluate==0.4.2 \              auto-gptq==0.7.1 optimum==1.21.4 awq==0.2.5 \              torch --extra-index-url https://download.pytorch.org/whl/cpu \              scipy>=1.11.0 numpy>=1.24.0 psutil>=5.9.0 pandas>=2.2.0 matplotlib>=3.8.0 tqdm>=4.66.0
python - << 'PY'
import transformers, datasets, evaluate, pandas, psutil, torch, numpy
print("Installed OK.")
PY


## 3) Create project structure & helper scripts

In [ ]:

import os, json, textwrap, pathlib
ROOT = "/content/tinyllm-phase2"
os.makedirs(ROOT, exist_ok=True)
for sub in ["scripts","models","data","results/charts"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)
print("Created:", ROOT)

# quantize_gptq.py
open(f"{ROOT}/scripts/quantize_gptq.py","w").write(r'''#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

def get_calib_texts(n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:2048]")
    return [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-gptq-4bit")
    ap.add_argument("--bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    base = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype="auto", device_map="auto")

    calib = get_calib_texts(1024)
    examples = [{"input_ids": tok(t, return_tensors="pt")["input_ids"]} for t in calib]

    qcfg = BaseQuantizeConfig(bits=args.bits, group_size=args.group_size, damp_percent=0.01, desc_act=True)
    qmodel = AutoGPTQForCausalLM.from_pretrained(base, quantize_config=qcfg)
    qmodel.quantize(examples)

    qmodel.save_pretrained(args.out)
    tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "quant_info.json"), "w") as f:
        json.dump({"method": "gptq", "bits": args.bits, "group_size": args.group_size}, f, indent=2)
    print("[GPTQ] Saved to", args.out)

if __name__ == "__main__":
    main()
''')

# quantize_awq.py
open(f"{ROOT}/scripts/quantize_awq.py","w").write(r'''#!/usr/bin/env python
import argparse, json, os
from datasets import load_dataset
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM

def get_calib_texts(n=1024):
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:2048]")
    return [x["text"] for x in ds if x["text"] and x["text"].strip()][:n]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out", default="models/phi3mini-awq-4bit")
    ap.add_argument("--w_bits", type=int, default=4)
    ap.add_argument("--group_size", type=int, default=128)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    model = AutoAWQForCausalLM.from_pretrained(args.model, torch_dtype="auto", device_map="auto")

    calib = get_calib_texts(1024)
    model.quantize(tokenizer=tok, calib_texts=calib, w_bits=args.w_bits, q_group_size=args.group_size, zero_point=True, version="GEMM")

    model.save_quantized(args.out, merge_lora=False, safetensors=True)
    tok.save_pretrained(args.out)
    with open(os.path.join(args.out, "quant_info.json"), "w") as f:
        json.dump({"method": "awq", "bits": args.w_bits, "group_size": args.group_size}, f, indent=2)
    print("[AWQ] Saved to", args.out)

if __name__ == "__main__":
    main()
''')

# evaluate.py
open(f"{ROOT}/scripts/evaluate.py","w").write(r'''#!/usr/bin/env python
import argparse, os, time, json, math, psutil, gc, pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline

RESULTS = Path("results")
DATA = Path("data")
RESULTS.mkdir(parents=True, exist_ok=True)

def ensure_eval_slices(seed=42):
    import random, json
    random.seed(seed)
    try:
        ds = load_dataset("hendrycks_test", "abstract_algebra", split="test")
        rows = []
        for ex in random.sample(list(ds), k=min(50, len(ds))):
            rows.append({"question": ex["question"], "choices": ex["choices"], "answer": ex["answer"]})
        with open(DATA/"eval_mmlu_50.jsonl", "w") as f:
            for r in rows: f.write(json.dumps(r)+"\\n")
    except Exception:
        pass
    try:
        ds = load_dataset("ai2_arc", "ARC-Easy", split="validation")
        rows = []
        for ex in random.sample(list(ds), k=min(50, len(ds))):
            rows.append({"question": ex["question"], "choices": ex["choices"]["text"], "answer": ex["answerKey"]})
        with open(DATA/"eval_arc_easy_50.jsonl", "w") as f:
            for r in rows: f.write(json.dumps(r)+"\\n")
    except Exception:
        pass
    try:
        ds = load_dataset("gsm8k", "main", split="test[:25]")
        rows = [{"question": r["question"], "answer": r["answer"]} for r in ds]
        with open(DATA/"eval_gsm8k_25.jsonl", "w") as f:
            for r in rows: f.write(json.dumps(r)+"\\n")
    except Exception:
        pass

def ppl_on_wikitext(model, tok, which="wikitext-2-raw-v1"):
    ds = load_dataset("wikitext", which, split="validation")
    enc = tok("\\n\\n".join(ds["text"]), return_tensors="pt")
    input_ids = enc["input_ids"]
    stride = 2048
    lls = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, input_ids.size(1), stride)):
            begin_loc = max(i + stride - 2048, 0)
            end_loc = min(i + stride, input_ids.size(1))
            trg_len = end_loc - i
            input_ids_slice = input_ids[:, begin_loc:end_loc]
            target_ids = input_ids_slice.clone()
            target_ids[:, :-trg_len] = -100
            out = model(input_ids_slice, labels=target_ids)
            lls.append(out.loss * trg_len)
    ppl = torch.exp(torch.stack(lls).sum() / end_loc).item()
    return ppl

def time_gen(pipe, prompt="Explain quantization in one paragraph.", gen_tokens=128):
    import time, psutil
    start_mem = psutil.Process().memory_info().rss
    t0 = time.time()
    _ = pipe(prompt, max_new_tokens=gen_tokens, do_sample=False)
    t1 = time.time()
    end_mem = psutil.Process().memory_info().rss
    total_time = t1 - t0
    tok_s = gen_tokens / total_time if total_time > 0 else float("nan")
    ms_per_tok = (total_time / gen_tokens) * 1000.0
    peak_ram_gb = max(start_mem, end_mem) / (1024**3)
    return {"latency_ms_per_token": ms_per_tok, "throughput_tok_s": tok_s, "peak_ram_gb": peak_ram_gb}

def load_model(which, model_name=None, model_dir=None):
    if which == "fp16":
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
        return model, tok
    elif which in ("gptq","awq"):
        tok = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype="auto", device_map="auto")
        return model, tok
    else:
        raise ValueError("which must be fp16|gptq|awq")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--which", required=True, choices=["fp16","gptq","awq"])
    ap.add_argument("--model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--model_dir", default=None)
    ap.add_argument("--wiki", default="wikitext-2-raw-v1", choices=["wikitext-2-raw-v1","wikitext-103-raw-v1"])
    args = ap.parse_args()

    ensure_eval_slices()

    model, tok = load_model(args.which, model_name=args.model, model_dir=args.model_dir)

    ppl = ppl_on_wikitext(model, tok, args.wiki)
    pipe = TextGenerationPipeline(model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)
    perf = time_gen(pipe)

    row = {
        "method": args.which,
        "model_dir_or_name": args.model if args.which=="fp16" else args.model_dir,
        "ppl": ppl,
        "peak_ram_gb": perf["peak_ram_gb"],
        "throughput_tok_s": perf["throughput_tok_s"],
        "latency_ms_per_token": perf["latency_ms_per_token"],
    }
    import pandas as pd, os
    os.makedirs("results", exist_ok=True)
    csv_path = "results/metrics.csv"
    df = pd.DataFrame([row])
    if os.path.exists(csv_path):
        old = pd.read_csv(csv_path)
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print("[Eval] Wrote", csv_path)

    # simple plots
    import matplotlib.pyplot as plt
    full = pd.read_csv(csv_path)
    os.makedirs("results/charts", exist_ok=True)
    plt.figure()
    full.groupby("method")["ppl"].mean().plot(kind="bar", title="PPL (lower is better)")
    plt.tight_layout(); plt.savefig("results/charts/accuracy_vs_size.png"); plt.close()

    plt.figure()
    full.groupby("method")["throughput_tok_s"].mean().plot(kind="bar", title="Throughput (tok/s)")
    plt.tight_layout(); plt.savefig("results/charts/speed_vs_ram.png"); plt.close()
    print("[Eval] Charts saved in results/charts")

if __name__ == "__main__":
    main()
''')

# make_plots.py
open(f"{ROOT}/scripts/make_plots.py","w").write(r'''#!/usr/bin/env python
import pandas as pd, matplotlib.pyplot as plt, os
df = pd.read_csv("results/metrics.csv")
os.makedirs("results/charts", exist_ok=True)
plt.figure(); df.groupby("method")["ppl"].mean().plot(kind="bar", title="PPL (lower is better)"); plt.tight_layout(); plt.savefig("results/charts/accuracy_vs_size.png"); plt.close()
plt.figure(); df.groupby("method")["throughput_tok_s"].mean().plot(kind="bar", title="Throughput (tok/s)"); plt.tight_layout(); plt.savefig("results/charts/speed_vs_ram.png"); plt.close()
print("Saved charts to results/charts/")
''')

# export_to_gguf.py with single-quoted HELP
open(f"{ROOT}/scripts/export_to_gguf.py","w").write(r'''#!/usr/bin/env python
HELP = '''Steps:
1) git clone https://github.com/ggerganov/llama.cpp
2) cd llama.cpp && make -j
3) python ./convert-hf-to-gguf.py --model <HF_ID_OR_DIR> --outfile <OUT.gguf> --outtype f16
4) ./quantize <OUT.gguf> <OUT-q4_0.gguf> q4_0
5) ./main -m <OUT-q4_0.gguf> -p "Explain quantization in 2 sentences." -n 128 --threads $(nproc)
'''
def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--hf_model", default="microsoft/Phi-3-mini-4k-instruct")
    ap.add_argument("--out_gguf", default="/content/tinyllm-phase2/models/phi3mini-f16.gguf")
    ap.add_argument("--llama_dir", default="/content/llama.cpp")
    args = ap.parse_args()
    print(HELP)
    print("Suggested command:")
    print(f"python {args.llama_dir}/convert-hf-to-gguf.py --model {args.hf_model} --outfile {args.out_gguf} --outtype f16")

if __name__ == "__main__":
    main()
''')

print("Scripts written.")


## 4) Quantize models (GPTQ & AWQ, 4-bit)

In [ ]:

%cd /content/tinyllm-phase2
!python scripts/quantize_gptq.py --model microsoft/Phi-3-mini-4k-instruct --out models/phi3mini-gptq-4bit --bits 4 --group_size 128
!python scripts/quantize_awq.py  --model microsoft/Phi-3-mini-4k-instruct --out models/phi3mini-awq-4bit  --w_bits 4 --group_size 128


## 5) Evaluate FP16 vs GPTQ vs AWQ (PPL + CPU perf)

In [ ]:

!python scripts/evaluate.py --which fp16 --model microsoft/Phi-3-mini-4k-instruct --wiki wikitext-2-raw-v1
!python scripts/evaluate.py --which gptq --model_dir models/phi3mini-gptq-4bit --wiki wikitext-2-raw-v1
!python scripts/evaluate.py --which awq  --model_dir models/phi3mini-awq-4bit  --wiki wikitext-2-raw-v1
!python scripts/make_plots.py


## 6) Export to GGUF & run with llama.cpp (CPU)

In [ ]:

%cd /content
!git clone -q https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make -j
%cd /content/tinyllm-phase2
!python scripts/export_to_gguf.py --hf_model microsoft/Phi-3-mini-4k-instruct --out_gguf models/phi3mini-f16.gguf --llama_dir /content/llama.cpp
print("Now run the printed convert command, then:")
print("  ./quantize models/phi3mini-f16.gguf models/phi3mini-q4_0.gguf q4_0")
print("  ./main -m models/phi3mini-q4_0.gguf -p 'Explain quantization in 2 sentences.' -n 128 --threads $(nproc)")


## 7) (Optional) Save artifacts to GCS

In [ ]:

%%bash
set -e
if [ -z "$BUCKET" ] or [ "$BUCKET" = "gs://YOUR_BUCKET" ]; then
  echo "Skip upload: set BUCKET variable first."; exit 0
fi
echo "Uploading to $ARTIFACTS"
gsutil -m cp -r /content/tinyllm-phase2/models $ARTIFACTS/
gsutil -m cp -r /content/tinyllm-phase2/results $ARTIFACTS/


## 8) Notes & troubleshooting
- If Phi-3 → GGUF conversion is finicky, swap to `mistralai/Mistral-7B-Instruct-v0.2`.
- Keep eval subsets small on CPU.
- For your report, use `results/metrics.csv` + charts in `results/charts/`.